# 🔬 AI Supply Chain — Lead-Lag Playground

Pick **two companies** and an **X variable** (candidate *leader*) and **Y variable** (candidate *follower*), run one function, and get a **statistics table**, a **cross-correlation table**, and a **4-panel figure**.

**The one function you call:**
```python
res = leadlag_experiment(x_company, x_var, y_company, y_var, max_lag=4, transform="yoy_z")
```
- `X = (x_company, x_var)` is the candidate **leader**; `Y = (y_company, y_var)` the **follower**.
- X and Y may be **different companies**, or the **same company with two variables** (e.g. does capex lead revenue?).
- `transform`: `yoy_z` (YoY growth, z-scored — default) · `level_z` (z-scored levels) · `yoy` (raw YoY).

**Data:** `financials.db` (real quarterly financials, 2016→2027). **Engine:** `pair_experiment.py` + `leadlag.py`.

> Interpretation guardrail: peak lag is chosen by strongest **positive** co-movement (demand propagation), and Granger causality is checked **both directions** to confirm the arrow. Lead-lag is *statistical precedence*, not a proven contract — cross-check top hits against SPLC / 10-K disclosures.

In [ ]:
# --- setup ---
%matplotlib inline
import pandas as pd
from IPython.display import display
from pair_experiment import leadlag_experiment, list_companies, AVAILABLE_VARS

pd.set_option('display.max_rows', 200)
print('Available X/Y variables:')
display(pd.DataFrame(AVAILABLE_VARS.items(), columns=['variable (use this)', 'meaning']))

In [ ]:
# --- browse companies you can use (ticker = what you pass in) ---
companies = list_companies()
print(f'{len(companies)} companies with quarterly data. Filter by segment, e.g.:')
display(companies[companies.segments.isin(['ai_chip','dram','hw_equipment','energy'])])

## ✏️ INPUTS — edit these, then run the cell below

| slot | meaning | example |
|---|---|---|
| `X_COMPANY`, `X_VAR` | candidate **leader** (company, variable) | `"NVDA"`, `"revenue"` |
| `Y_COMPANY`, `Y_VAR` | candidate **follower** (company, variable) | `"MU"`, `"revenue"` |
| `MAX_LAG` | how many quarters to scan each side | `4` |
| `TRANSFORM` | `"yoy_z"` / `"level_z"` / `"yoy"` | `"yoy_z"` |

In [ ]:
# ============== EDIT THESE ==============
X_COMPANY = "NVDA"            # candidate leader company (ticker)
X_VAR     = "revenue"  # candidate leader variable
Y_COMPANY = "MU"             # candidate follower company (ticker)
Y_VAR     = "revenue"  # candidate follower variable
MAX_LAG   = 4                # quarters scanned on each side
TRANSFORM = "yoy_z"          # 'yoy_z' | 'level_z' | 'yoy'
# =======================================

res = leadlag_experiment(X_COMPANY, X_VAR, Y_COMPANY, Y_VAR,
                         max_lag=MAX_LAG, transform=TRANSFORM)

print('VERDICT:', res.verdict, '\n')
print('— Statistics —')
display(res.stats_table())
print('— Cross-correlation function (k>0 ⇒ X leads Y) —')
display(res.ccf_table())
res.plot();

## More things to try

Just edit the inputs cell and re-run. A few illustrative experiments:

```python
# AI-chip demand -> memory revenue (the headline edge)
leadlag_experiment("AVGO", "revenue", "MU", "revenue")

# fab-equipment peers
leadlag_experiment("AMAT", "revenue", "LRCX", "revenue")

# intra-company: does a hyperscaler's capex lead its own revenue?
leadlag_experiment("MSFT", "capex", "MSFT", "revenue")

# does NVIDIA revenue lead a downstream server OEM's inventory build?
leadlag_experiment("NVDA", "revenue", "DELL", "inventory")
```

**Reading the outputs**
- **best lag k\*** with **k>0** ⇒ X leads Y by k quarters (k<0 ⇒ Y leads X).
- **peak corr p-value < 0.05** ⇒ the co-movement at that lag is statistically significant.
- **Granger X→Y p < Y→X p** (and < 0.05) ⇒ the causal arrow is confirmed in the X→Y direction.
- **transmission β** ≈ pass-through strength; **R²** = fit quality of the lagged regression.

**Caveats:** ~24–34 quarters per name → correlation is primary, Granger secondary. `level_z` on trending series can inflate correlations (spurious trend); `yoy_z` is the safer default.